In [6]:
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options

import time
import json
from datetime import datetime

In [7]:
TARGET_URL = "https://www.topcv.vn/tim-viec-lam-moi-nhat-tai-ho-chi-minh-l2?exp=1&type_keyword=0&sba=1&locations=l2"
# Đặt tên file output dựa trên thời gian chạy để tránh ghi đè
TIMESTAMP_STR = datetime.now().strftime('%Y%m%d_%H%M%S')
HTML_OUTPUT_FILENAME = f"topcv_selenium_page_{TIMESTAMP_STR}.html"
JSON_OUTPUT_FILENAME = f"topcv_jobs_data_{TIMESTAMP_STR}.json"

print(f"URL Mục tiêu: {TARGET_URL}")
print(f"File HTML sẽ được lưu là: {HTML_OUTPUT_FILENAME}")
print(f"File JSON sẽ được lưu là: {JSON_OUTPUT_FILENAME}")

URL Mục tiêu: https://www.topcv.vn/tim-viec-lam-moi-nhat-tai-ho-chi-minh-l2?exp=1&type_keyword=0&sba=1&locations=l2
File HTML sẽ được lưu là: topcv_selenium_page_20250520_201720.html
File JSON sẽ được lưu là: topcv_jobs_data_20250520_201720.json


# Configure and Initialize Selenium WebDriver


In [8]:

print("Đang cấu hình Chrome Options ........")
chrome_options = Options()

# --- Các tùy chọn quan trọng ---
# 1. Chạy ở chế độ "headless" (không mở cửa sổ trình duyệt vật lý)
#    Khi debug, bạn nên comment dòng này lại để có thể thấy trình duyệt hoạt động.
# chrome_options.add_argument("--headless") 

# 2. Các cờ thường cần thiết khi chạy trong môi trường Linux hoặc container (như Docker)
chrome_options.add_argument("--no-sandbox")
chrome_options.add_argument("--disable-dev-shm-usage") # Khắc phục lỗi liên quan đến /dev/shm khi thiếu tài nguyên

# 3. Tắt GPU, thường không cần thiết cho scraping và có thể cải thiện hiệu suất/ổn định
chrome_options.add_argument("--disable-gpu")

# 4. Đặt User-Agent giống như trình duyệt thông thường để tránh bị phát hiện là bot
#    Bạn có thể thay bằng User-Agent từ trình duyệt của mình nếu muốn
chrome_options.add_argument("user-agent=Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/136.0.0.0 Safari/537.36")

# 5. Đặt kích thước cửa sổ (có thể quan trọng đối với một số trang web responsive)
chrome_options.add_argument("--window-size=1920,1080")

# 6. (Tùy chọn) Chạy ở chế độ ẩn danh
# chrome_options.add_argument("--incognito")

# 7. (Tùy chọn) Tắt thông báo "Chrome is being controlled by automated test software"
chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])
chrome_options.add_experimental_option('useAutomationExtension', False)

# 8. (Tùy chọn) Tắt hình ảnh để tải trang nhanh hơn (nếu không cần hình ảnh)
# chrome_options.add_argument('--blink-settings=imagesEnabled=false')

print("Chrome Options đã được cấu hình.")

# --- Khởi tạo WebDriver ---
# Selenium 4 sẽ tự động tìm chromedriver nếu nó nằm trong PATH hệ thống.
# Nếu bạn đặt chromedriver ở một vị trí cụ thể và không có trong PATH, bạn cần dùng Service:
# service = Service(executable_path='/duong/dan/toi/chromedriver')
# driver = webdriver.Chrome(service=service, options=chrome_options)

try:
    print("Đang khởi tạo WebDriver...")
    driver = webdriver.Chrome(options=chrome_options)
    print("WebDriver đã được khởi tạo thành công!")
except Exception as e:
    print(f"Lỗi khi khởi tạo WebDriver: {e}")
    print("Hãy đảm bảo ChromeDriver đã được cài đặt đúng và phiên bản tương thích với Chrome của bạn.")
    print("Nếu ChromeDriver không nằm trong PATH, bạn cần chỉ định 'executable_path' thông qua 'Service'.")
    driver = None # Đặt driver là None nếu khởi tạo lỗi

# Để đảm bảo driver được đóng nếu notebook bị ngắt đột ngột hoặc cell này chạy lại nhiều lần
# chúng ta sẽ lưu trữ driver vào một biến global (hơi xấu nhưng tiện cho notebook)
# hoặc tốt hơn là quản lý nó cẩn thận trong các cell sau.
# Hiện tại, chúng ta sẽ đóng nó ở cell cuối cùng.

Đang cấu hình Chrome Options ........
Chrome Options đã được cấu hình.
Đang khởi tạo WebDriver...
WebDriver đã được khởi tạo thành công!


# Navigate to Target URL, Wait, and Get Page Source


In [9]:
#Kiểm tra xem biến drivẻ từ cell trước có tồn tại và không phải là None không

if 'driver' in locals() and driver is not None:
    try:
        print(f"Đang điều khiển trình duyệt truy cập URL: {TARGET_URL}")
        driver.get(TARGET_URL) # Lệnh để trình duyệt mở URL
        print(f"Đã gửi yêu cầu tải trang. URL trên trình duyệt tên là: {driver.current_url}")
        
        # Chờ một khoảng thời gian cố định để trang tải hoàn chỉnh (bao gồm JavaScript)
        # Đối với các trang phức tạp, ta có thể cần phải tăng thời gian này
        # hoặc sử dụng WebDriverWait để đợi một element cụ thể xuất hiện.
        wait_time_seconds = 14 # Thử với 14 giây, có thể tăng nếu cần
        print(f"Đang đợi {wait_time_seconds} giây để nội dung trang tải hoàn chỉnh...")
        time.sleep(wait_time_seconds)
        print("Thời gian chờ đã kết thúc.")
        
        # Lấy mã nguồn HTML của trang sau khi đã tải và JavaScript (nếu có) đã chạy
        print("Đang lấy mã nguồn HTML của trang (page_source)...")
        page_source_html = driver.page_source
        
        if page_source_html:
            print(f"Đã lấy được page_source HTML (kích thước: {len(page_source_html)} bytes).")
            
            # Lưu nội dung HTML ra file để kiểm tra và tìm selector sau này
            with open(HTML_OUTPUT_FILENAME, "w", encoding="utf-8") as f:
                f.write(page_source_html)
            print(f"Nội dung HTML của trang TopCV đã được lưu vào file: '{HTML_OUTPUT_FILENAME}'")
            print("Hãy mở file này bằng trình duyệt để kiểm tra nội dung và chuẩn bị cho việc tìm selector.")
        else:
            print("Không lấy được page_source HTML. Trang có thể chưa tải xong hoặc có lỗi.")
    except Exception as e:
        print(f"Lỗi trong quá trình truy cập URL hoặc lấy page_source: {e}")
        # Cân nhắc đóng driver nếu có lỗi nghiêm trọng ở đây, 
        # nhưng chúng ta sẽ đóng ở cell cuối cùng để đảm bảo.
else:
    print("Biến 'driver' không tồn tại hoặc chưa được khởi tạo. Hãy chạy lại Cell 2.")

Đang điều khiển trình duyệt truy cập URL: https://www.topcv.vn/tim-viec-lam-moi-nhat-tai-ho-chi-minh-l2?exp=1&type_keyword=0&sba=1&locations=l2
Đã gửi yêu cầu tải trang. URL trên trình duyệt tên là: https://www.topcv.vn/tim-viec-lam-moi-nhat-tai-ho-chi-minh-l2?exp=1&type_keyword=0&sba=1&locations=l2
Đang đợi 14 giây để nội dung trang tải hoàn chỉnh...
Thời gian chờ đã kết thúc.
Đang lấy mã nguồn HTML của trang (page_source)...
Đã lấy được page_source HTML (kích thước: 1441776 bytes).
Nội dung HTML của trang TopCV đã được lưu vào file: 'topcv_selenium_page_20250520_201720.html'
Hãy mở file này bằng trình duyệt để kiểm tra nội dung và chuẩn bị cho việc tìm selector.


# Parse HTML with BeautifulSoup and Extract Data

In [ ]:
# Đảm bảo biến page_source_html được định nghĩa và không phải là None
if 'page_source_html' in locals() and page_source_html is not None:
    print(f"Đang thực hiện phân tích (Parse) HTML với (kích thước là: {len(page_source_html)} bytes)...")
    soup = BeautifulSoup(page_source_html, 'html.parser')
    
    all_jobs_data = [] # Danh sách để lưu trữ tất cả các jobs đã cào được
    
    # 1. Tìm container chính chứa tất cả các job items
    # Dựa trên phân tích HTML rằng là, chúng ta có class="job-list-main" có con là class="box-job-list"
    job_list_main_container = soup.find('div', class_='job-list-main')
    job_list_container = None
    if job_list_main_container:
        job_list_container = job_list_main_container.find('div', class_='box-plot-list')
        
    if job_list_container:
        # 2. Tìm tất cả các job items bên trong container đó
        # Dựa trên phân tích của HTML mà đã phân tích trước đó, mình có class="job-item-search-result"
        job_item_elements = job_list_container.find_all('div', class_='job-item-search-result')
        print(f"Tìm thấy tổng cộng {len(job_item_elements)} job items trong 'div.box-job-list'.")
        